1. Data Pipeline

In [21]:
from torchvision import datasets,transforms
from torch.utils.data import DataLoader

1.1 Normalize

In [22]:
transform=transforms.ToTensor()

1.2 Load the MNIST dataset

In [23]:
train_dataset=datasets.MNIST(
    root="/.data",
    train=True,
    download=True,
    transform=transform
)
test_dataset=datasets.MNIST(
    root="/.data",
    train=False,
    download=True,
    transform=transform
)
train_loader=DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)
test_loader=DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=False
)

2. Architecture Design

In [24]:
import torch
import torch.nn as nn

2.1 MLP design

In [25]:
class MNIST_MLP(nn.Module):
  def __init__(self):
    super().__init__()
    self.network=nn.Sequential(
        nn.Flatten(),
        nn.Linear(28*28,128),
        nn.ReLU(),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Linear(64,10)
    )
  def forward(self,x):
    return self.network(x)



2.2 Cross entropy loss function and Optimizer

In [26]:
model= MNIST_MLP()

In [27]:
criterion=nn.CrossEntropyLoss()

In [28]:
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

3. Training

In [29]:
num_epochs=4

In [30]:
for epoch in range(num_epochs):
  model.train()
  running_loss=0.0
  correct=0
  total=0
  for images,labels in train_loader:
    outputs=model(images)
    loss=criterion(outputs,labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    running_loss+=loss.item()
    _,predicted=torch.max(outputs,1)
    total+=labels.size(0)
    correct+=(predicted==labels).sum().item()

  epoch_loss=running_loss/len(train_loader)
  epoch_accuracy=correct*100/total
  print(f"Epoch[{epoch+1}/{num_epochs}]"
  f"Loss:{epoch_loss:.4f}"
  f"Accuracy:{epoch_accuracy:.2f}%")



Epoch[1/4]Loss:0.3495Accuracy:90.36%
Epoch[2/4]Loss:0.1474Accuracy:95.59%
Epoch[3/4]Loss:0.0979Accuracy:97.06%
Epoch[4/4]Loss:0.0731Accuracy:97.75%


3.1 Testing

In [31]:
model.eval()
test_loss=0
correct=0
total=0
with torch.no_grad():
  for images,labels in test_loader:
    outputs=model(images)
    loss=criterion(outputs,labels)
    test_loss+=loss.item()
    _,predicted=torch.max(outputs,1)
    total+=labels.size(0)
    correct+=(predicted==labels).sum().item()
test_loss=test_loss/len(test_loader)
test_accuracy=correct*100/total
print(f"Test Loss:{test_loss:.4f}\n"
  f"Test Accuracy:{test_accuracy:.2f}%")


Test Loss:0.0527
Test Accuracy:98.43%


4. Custom Inference Pipeline

In [39]:
images,labels=next(iter(test_loader))
image=images[0]

In [40]:
def predict_digit(image):
  model.eval()
  outputs=model(image)
  probabilities=torch.softmax(outputs,dim=1)
  predicted=torch.argmax(probabilities,dim=1).item()
  return predicted,probabilities[0]

In [41]:
digit,probabilities=predict_digit(image)
print("predicted_digit:",digit)
for i,probability in enumerate(probabilities):
  print(f"{i}:{probability.item():.4f}")

predicted_digit: 5
0:0.0000
1:0.0000
2:0.0000
3:0.0031
4:0.0000
5:0.9969
6:0.0000
7:0.0000
8:0.0000
9:0.0000
